In [ ]:
!pip install transformers datasets

In [ ]:
from datasets import load_dataset
import numpy as np

from dotenv import load_dotenv
import os

load_dotenv()
from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"))


In [ ]:
import socket, urllib3.util.connection
urllib3.util.connection.allowed_gai_family = lambda: socket.AF_INET

In [ ]:
raw_datasets = load_dataset("glue", "sst2")

In [ ]:
raw_datasets

In [ ]:
raw_datasets['train'].data

In [ ]:
raw_datasets['train'][0]

In [ ]:
from transformers import AutoTokenizer
# checkpoint = "bert-base-uncased"
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
tokenized_sentences = tokenizer(raw_datasets['train'][0:3]['sentence'])
from pprint import pprint
pprint(tokenized_sentences)

In [ ]:
def tokenize_fn(batch):
  return tokenizer(batch['sentence'], truncation=True)
tokenized_datasets = raw_datasets.map(tokenize_fn, batched=True)

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
  'my_trainer',
  evaluation_strategy='epoch',
  save_strategy='epoch',
  num_train_epochs=1,
)

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2)

In [ ]:
import transformers
type(model)

In [ ]:
model

In [ ]:
!pip install torchinfo


In [ ]:
from torchinfo import summary
summary(model)

In [ ]:
params_before = []
for name, p in model.named_parameters():
  params_before.append(p.detach().cpu().numpy())

In [ ]:
!pip install evaluate

In [ ]:
from transformers import Trainer
import evaluate
# datasets.load_metric was moved to the separate `evaluate` package.
# Provide a compatibility alias so the existing `load_metric(...)` call still works.
load_metric = evaluate.load
metric = load_metric("glue", "sst2")

In [ ]:
metric.compute(predictions=[1, 0, 1], references=[1, 0, 0])

In [ ]:
def compute_metrics(logits_and_labels):
  # metric = load_metric("glue", "sst2")
  logits, labels = logits_and_labels
  predictions = np.argmax(logits, axis=-1)
  return metric.compute(predictions=predictions, references=labels)


trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# trainer.train()

In [ ]:
trainer.save_model('my_saved_model')


In [ ]:
from transformers import pipeline
newmodel = pipeline('text-classification', model='my_saved_model', device=0)

In [ ]:
newmodel('This movie is great!')
# [{'label': 'LABEL_1', 'score': 0.9995631575584412}]
newmodel('This movie sucks')
# [{'label': 'LABEL_0', 'score': 0.9965962767601013}]

In [ ]:
!cat my_saved_model/config.json


In [ ]:
# later he confirmed that diff of trainable params is non 0 which indicates that model was finetuned